In [ ]:
# =====================================================
# 1. RUTAS 
# =====================================================

ruta_puntos =r"D:\IMAGEN SATELITAL CAJAMARCA\Viviendas_puntos\CENTROS_POBLADOS_2017.shp"       
#"D:\IMAGEN SATELITAL CAJAMARCA\Viviendas_puntos\VIVIENDAS_RURALES_2017.shp"
#"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\viviendas_filtradas.shp"
ruta_raster = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CO_2510221339013\VOL_PER1_ORT_001_002705\IMG_PER1_ORT_PMS_002705\IMG_PER1_20210706153514_ORT_PMS_002705.TIF"       # <-- cambia aquí
ruta_salida = r"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\CCPP\CCPP_filtradas.shp"
#D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\SIN_NODATA

## RECORTE

In [1]:
import geopandas as gpd
import rasterio
import numpy as np
import os

# =====================================================
# RUTAS
# =====================================================

ruta_puntos = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\Viviendas_puntos\CENTROS_POBLADOS_2017.shp"
#"D:\Imagen_Sat_F\CAJAMARCA DIEGO\dentro_AOI_IMAGEN\DENTRO_VIVIENDAS_CCPP2017\CCPP_filtradas.shp"
ruta_raster = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CO_2510221339013\VOL_PER1_ORT_001_002705\IMG_PER1_ORT_PMS_002705\IMG_PER1_20210706153514_ORT_PMS_002705.TIF"       # <-- cambia aquí
ruta_salida = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\Viviendas_puntos\dentro_de _la imagen_\viviendas_dentro_real.shp"

# =====================================================
# LIMPIAR SALIDA
# =====================================================

os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
    f = ruta_salida.replace(".shp", ext)
    if os.path.exists(f):
        os.remove(f)

# =====================================================
# LEER PUNTOS
# =====================================================

puntos = gpd.read_file(ruta_puntos)

# =====================================================
# LEER RASTER
# =====================================================

with rasterio.open(ruta_raster) as src:

    raster_crs = src.crs
    bandas = src.count

    if puntos.crs != raster_crs:
        puntos = puntos.to_crs(raster_crs)

    coords = [(geom.x, geom.y) for geom in puntos.geometry]

    # Leer TODAS las bandas en cada punto
    valores = np.array([val for val in src.sample(coords)])

# =====================================================
# FILTRAR DONDE HAYA INFORMACIÓN REAL
# =====================================================

# Mantener puntos donde NO todas las bandas sean 0
mascara_valida = ~(np.all(valores == 0, axis=1))

puntos_validos = puntos[mascara_valida].copy()

print("Puntos originales:", len(puntos))
print("Puntos dentro reales:", len(puntos_validos))

# =====================================================
# EXPORTAR
# =====================================================

puntos_validos.to_file(
    ruta_salida,
    driver="ESRI Shapefile",
    index=False
)

print("Proceso terminado correctamente ✅")

Puntos originales: 915
Puntos dentro reales: 91
Proceso terminado correctamente ✅


## MST

In [7]:
import geopandas as gpd
import numpy as np
from shapely.geometry import MultiPoint
from shapely.ops import unary_union
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
import networkx as nx

# -----------------------------
# 1. Cargar capa de viviendas
# -----------------------------
input_path = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CENTRIODE_POLI\resultado_final.shp"
#"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\SIN_NODATA\viviendas_dentro_real.shp"
gdf = gpd.read_file(input_path)

print("CRS:", gdf.crs)

# -----------------------------
# 2. Verificar CRS (debe estar en metros)
# -----------------------------
# Si está en lat/long, reproyectar:
# gdf = gdf.to_crs("EPSG:32718")

# -----------------------------
# 3. Agrupar por CODCCPP
# -----------------------------
centros = []

for cod, grupo in gdf.groupby("CODCCPP"):
    
    if len(grupo) < 3:
        continue  # evitar grupos muy pequeños
    
    coords = np.array([[p.x, p.y] for p in grupo.geometry])
    
    # -------------------------
    # 4. MST interno
    # -------------------------
    dist_matrix = squareform(pdist(coords))
    mst_sparse = minimum_spanning_tree(dist_matrix)
    mst_matrix = mst_sparse.toarray()
    
    G = nx.Graph()
    
    for i in range(len(coords)):
        for j in range(len(coords)):
            if mst_matrix[i, j] > 0:
                G.add_edge(i, j, weight=mst_matrix[i, j])
    
    # -------------------------
    # 5. Corte opcional
    # -------------------------
    threshold = 256  # ajustar según rural/urbano
    
    edges_to_remove = [(u, v) for u, v, d in G.edges(data=True)
                       if d["weight"] > threshold]
    
    G.remove_edges_from(edges_to_remove)
    
    # -------------------------
    # 6. Unir puntos del grupo
    # -------------------------
    multipoint = MultiPoint(grupo.geometry.tolist())
    
    # Buffer para generar polígono
    poligono = multipoint.buffer(40)  # 40 metros
    
    centros.append({
        "CODCCPP": cod,
        "geometry": poligono
    })

# -----------------------------
# 7. Crear GeoDataFrame final
# -----------------------------
gdf_centros = gpd.GeoDataFrame(centros, crs=gdf.crs)

# -----------------------------
# 8. Exportar a GPKG
# -----------------------------
output_path = "centros_poblados.gpkg"
gdf_centros.to_file(output_path, layer="centros_ccpp", driver="GPKG")

print("Proceso finalizado.")

CRS: EPSG:32717
Proceso finalizado.


In [ ]:
# =====================================================
# 1. RUTAS (EDITA SOLO ESTO)
# =====================================================

ruta_puntos =r"D:\IMAGEN SATELITAL CAJAMARCA\Viviendas_puntos\CENTROS_POBLADOS_2017.shp"        # <-- cambia aquí
#"D:\IMAGEN SATELITAL CAJAMARCA\Viviendas_puntos\VIVIENDAS_RURALES_2017.shp"
#"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\viviendas_filtradas.shp"
#input_path = r"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\SIN_NODATA\viviendas_dentro_real.shp"

ruta_raster = r"D:\IMAGEN SATELITAL CAJAMARCA\CAJAMARCA\IMAGEN SAT\VOL_PER1_ORT_001_000195-20260127T162812Z-3-001\VOL_PER1_ORT_001_000195\IMG_PER1_ORT_PMS_000195\IMG_PER1_20220907154232_ORT_PMS_000195.TIF"       # <-- cambia aquí
ruta_salida = r"D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\CCPP\CCPP_filtradas.shp"
#D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\SIN_NODATA

In [ ]:
# MST por CODCCPP y exportación a GPKG
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString
from collections import defaultdict, deque

# -----------------------------
# 1. Cargar capa de viviendas
# -----------------------------
ruta_entrada = r"C:\Users\decg112\Downloads\capas de cajamarca\vinculacion_ccpp.gpkg"
ruta_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\mst_centros_poblados.gpkg"
#D:\IMAGEN SATELITAL CAJAMARCA\dentro_imagen

gdf = gpd.read_file(ruta_entrada)

# Verificar que esté en UTM (proyectado en metros)
if not gdf.crs.is_projected:
    raise ValueError("La capa debe estar en coordenadas proyectadas (UTM en metros)")

# -----------------------------
# 2. Función para calcular MST (Prim)
# -----------------------------
def compute_mst(points):
    n = len(points)
    visited = [False] * n
    visited[0] = True
    edges = []

    for _ in range(n - 1):
        min_dist = float("inf")
        min_edge = None

        for i in range(n):
            if visited[i]:
                for j in range(n):
                    if not visited[j]:
                        dist = np.linalg.norm(points[i] - points[j])
                        if dist < min_dist:
                            min_dist = dist
                            min_edge = (i, j, dist)

        i, j, dist = min_edge
        visited[j] = True
        edges.append((i, j, dist))

    return edges

# -----------------------------
# 3. Procesar por CODCCPP
# -----------------------------
threshold = 200  # metros (ajustar según tu realidad)

lineas = []

for codccpp, grupo in gdf.groupby("CODCCPP"):

    if len(grupo) < 2:
        continue

    # Obtener coordenadas
    coords = np.array([[geom.x, geom.y] for geom in grupo.geometry])

    # Calcular MST
    edges = compute_mst(coords)

    # Cortar aristas largas
    filtered_edges = [(i, j) for i, j, d in edges if d < threshold]

    # Crear líneas
    for i, j in filtered_edges:
        linea = LineString([coords[i], coords[j]])
        lineas.append({
            "CODCCPP": codccpp,
            "geometry": linea
        })

# -----------------------------
# 4. Crear GeoDataFrame resultado
# -----------------------------
gdf_mst = gpd.GeoDataFrame(lineas, crs=gdf.crs)

# -----------------------------
# 5. Exportar a GPKG
# -----------------------------
gdf_mst.to_file(ruta_salida, layer="mst_codccpp", driver="GPKG")

print("MST exportado correctamente a:", ruta_salida)

## CENTROIDES DENTRO DEL POLIGONO

### atribuir poligonos 

In [5]:
import geopandas as gpd

# 1. Define las rutas de tus archivos
ruta_puntos = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\dentro_AOI_IMAGEN\DENTRO_VIVIENDAS_CCPP2017\viviendas_dentro_real.shp" 
#"D:\Imagen_Sat_F\CAJAMARCA DIEGO\dentro_AOI_IMAGEN\DENTRO_VIVIENDAS_CCPP2017\viviendas_dentro_real.shp"
ruta_poligonos = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CAPA_VIVIENDA\IMG_PER1_20210706153514_ORT_PMS_002705_class.gpkg"

#ruta_raster = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CO_2510221339013\VOL_PER1_ORT_001_002705\IMG_PER1_ORT_PMS_002705\IMG_PER1_20210706153514_ORT_PMS_002705.TIF"       # <-- cambia aquí


#"C:\Users\decg112\Downloads\capas de cajamarca\ultima_version_vivendas_IMG_PER1_20220907154232_ORT_PMS_000195.gpkg"
ruta_salida =r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CENTRIODE_POLI\resultado_final.shp"           #"ruta/a/tu/resultado_final.shp"
#D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\centroide_POLI

# 2. Cargar las capas
print("Cargando capas...")
puntos = gpd.read_file(ruta_puntos)
poligonos = gpd.read_file(ruta_poligonos)

# 3. Asegurar que tengan el mismo Sistema de Coordenadas (CRS)
if puntos.crs != poligonos.crs:
    poligonos = poligonos.to_crs(puntos.crs)

# 4. Encontrar el polígono más cercano a cada punto
print("Buscando el polígono más cercano...")
uniones = gpd.sjoin_nearest(puntos, poligonos, how='left')

# (Opcional pero recomendado) Evitar puntos duplicados si hay empates en distancia
uniones = uniones[~uniones.index.duplicated(keep='first')]

# 5. Hacer que el punto caiga DENTRO del polígono más cercano
print("Moviendo los puntos al interior del polígono...")
# Identificamos cuáles son los polígonos ganadores
poligonos_cercanos = poligonos.loc[uniones['index_right']]

# representative_point() asegura que el punto siempre caiga dentro de la figura (incluso en las "L")
puntos['geometry'] = poligonos_cercanos.representative_point().values

# 6. Guardar el resultado
puntos.to_file(ruta_salida)
print("¡Proceso terminado! Archivo guardado.")

Cargando capas...
Buscando el polígono más cercano...
Moviendo los puntos al interior del polígono...
¡Proceso terminado! Archivo guardado.


###  poligonos reales 

In [ ]:
import geopandas as gpd #ESTE E UN OEUABF

# 1. Define las rutas
ruta_puntos = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CENTRIODE_POLI\resultado_final.shp"
ruta_poligonos = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CAPA_VIVIENDA\edificios_regularizados.gpkg"
#"C:\Users\decg112\Downloads\capas de cajamarca\ultima_version_vivendas_IMG_PER1_20220907154232_ORT_PMS_000195.gpkg"
ruta_poligonos_salida = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CENTRIODE_POLI\poligonos_con_atributos.gpkg" # ¡Ahora guardaremos los polígonos!
#D:\IMAGEN SATELITAL CAJAMARCA\dentro_image\centroide_POLI\FINAL\poligonos_con_atributos.gpkg

# 2. Cargar las capas
print("Cargando capas...")
puntos = gpd.read_file(ruta_puntos)
poligonos = gpd.read_file(ruta_poligonos)

# 3. Asegurar que tengan el mismo Sistema de Coordenadas (CRS)
if puntos.crs != poligonos.crs:
    poligonos = poligonos.to_crs(puntos.crs)

# 4. Relacionar espacialmente para saber quién va con quién
print("Calculando relación espacial...")
uniones = gpd.sjoin_nearest(puntos, poligonos, how='left')

# En 'uniones', tenemos los datos del punto (como CODCCPP) y una columna llamada 'index_right' 
# que nos dice el ID del polígono más cercano. Extraemos esa tabla de relación:
relacion_atributos = uniones[['index_right', 'CODCCPP', 'UBIGEO']] # Puedes agregar UBIGEO si también lo quieres

# PRECAUCIÓN: Si por casualidad dos puntos cayeron en el mismo polígono, 
# eliminamos los duplicados para no clonar polígonos por error.
relacion_atributos = relacion_atributos.drop_duplicates(subset=['index_right'])

# 5. Transferir los atributos a los polígonos
print("Pasando atributos a los polígonos...")
poligonos_final = poligonos.merge(
    relacion_atributos,
    left_index=True,        # El índice de la capa de polígonos
    right_on='index_right', # El índice que guardamos en la relación
    how='left'              # 'left' asegura que no perdamos polígonos aunque no tengan un punto cercano
)

# Limpiamos la columna de índice auxiliar que ya no necesitamos
poligonos_final = poligonos_final.drop(columns=['index_right'])

# 6. Guardar el resultado
poligonos_final.to_file(ruta_poligonos_salida, driver="GPKG")
print("¡Proceso terminado! Tus polígonos ahora tienen el CODCCPP.")

Cargando capas...
Calculando relación espacial...
Pasando atributos a los polígonos...
¡Proceso terminado! Tus polígonos ahora tienen el CODCCPP.
